In [29]:
import cv2
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython import display as ipd
from pathlib import Path

In [2]:
from src.utils.utils import ComputerVision
from src.data.loaders import DatasetLoader

In [3]:
ROOT_DIR = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT_DIR / "src"))

print(ROOT_DIR)

/home/automation/master-degree-project


In [4]:
OUTPUT_DIR = ROOT_DIR / "outputs" / "notebooks" / "20260610-01-frames-brightness-analysis"
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_DIR = ROOT_DIR / "datasets" / "RLVS" / "Violence"
INPUT_DIR.mkdir(exist_ok=True)

In [5]:
cv = ComputerVision()
dataset_loader = DatasetLoader(INPUT_DIR)

In [6]:
video_files = dataset_loader.load_videos_files()
print(f"Found {len(video_files)} video files in {INPUT_DIR}")
print(f"Video files: {[video.name for video in video_files]}")

Found 20 video files in /home/automation/master-degree-project/datasets/RLVS/Violence
Video files: ['V_19.mp4', 'V_10.mp4', 'V_13.mp4', 'V_2.mp4', 'V_12.mp4', 'V_14.mp4', 'V_1.mp4', 'V_4.mp4', 'V_15.mp4', 'V_6.mp4', 'V_17.mp4', 'V_11.mp4', 'V_8.mp4', 'V_7.mp4', 'V_3.mp4', 'V_5.mp4', 'V_16.mp4', 'V_18.mp4', 'V_9.mp4', 'V_20.mp4']


In [ ]:
cap = cv2.VideoCapture(str(video_files[0]))
fps = cap.get(cv2.CAP_PROP_FPS)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis("off")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            continue

        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ipd.clear_output(wait=True)
        plt.show()
        time.sleep(1/fps)

except KeyboardInterrupt:
    cap.release()
    print("Stopped.")

<Figure size 640x480 with 0 Axes>

In [21]:
# release all captures when done
for cap in caps:
    cap.release()   

In [18]:
def annotate_video(input_path, output_path, thresholds):
    cap = cv2.VideoCapture(input_path)
    
    if not cap.isOpened():
        print("Error: Could not open the video file.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    frame_counts = {'Day': 0, 'Evening': 0, 'Night': 0}
    frame_index = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        brightness = cv.calculate_brightness(frame)
        classification = cv.classify_frame(brightness, thresholds)
        
        # print(f"Frame {frame_index}: Brightness={brightness:.2f}, Classification={classification}")
        
        frame_counts[classification] += 1
        text = f"Frame: {frame_index}, Time: {classification}"
        
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"FPS: {fps:.2f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
        
        out.write(frame)
        frame_index += 1
        
    cap.release()
    out.release()
    total_frames = sum(frame_counts.values())
    percentages = {key: (count / total_frames) * 100 for key, count in frame_counts.items()}
    print(f"Annotated video saved to {output_path}")
    print(f"Day: {percentages['Day']:.2f}%, Evening: {percentages['Evening']:.2f}%, Night: {percentages['Night']:.2f}%")

In [19]:
thresholds = {
    'day': 120,
    'evening': 70,
    'night': 0
}

input_video_path = INPUT_DIR / "V_20.mp4"
output_video_path = OUTPUT_DIR / "annotated_video.avi"
annotate_video(input_video_path, output_video_path, thresholds)

Annotated video saved to /home/automation/master-degree-project/outputs/notebooks/20260610-01-frames-brightness-analysis/annotated_video.avi
Day: 16.33%, Evening: 83.67%, Night: 0.00%
